In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_hydrophobic_sites)=
# get_hydrophobic_sites

*Recognizing hydrophobic atoms from declared chemical graphs.*

Inspect candidate atoms independently of geometry. This experimental definition reproduces ProLIF 2.2.2 SMARTS incorporating RDKit feature patterns. RDKit is an optional calculation dependency; ProLIF is a reference, not a runtime dependency. Residue hydrophobicity scales remain a separate interpretation.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_hydrophobic_sites` for arguments, returned data and explicit failures.
:::

## Basic usage

Let's show how this function works with aliphatic carbon, sulfur and fluorinated controls. Required source information includes elements, complete bonds, orders, formal charges and atom/bond aromatic flags. Recognition does not require coordinates.

In [2]:
import molsysmt as msm

In [3]:
from rdkit import Chem
molsys = msm.convert(Chem.MolFromSmiles('CCC.CSC.CF'), to_form='molsysmt.MolSys')
sites = msm.physchem.get_hydrophobic_sites(molsys)
print(sites['hydrophobic_atom_indices'])
print(sites['method'])

[1 4]
smarts_hydrophobic_atoms


## Understanding chemical membership

This is not an all-carbon mask. The profile includes neutral aromatic c/s, Br/I, sulfur with no hydrogens and valence two, and specified aliphatic carbon environments. Carbon bonded to N/O/F is excluded, even when aromatic. Terminal methyl carbons do not match generically. F and Cl themselves do not match. Every returned value is a source atom index, not an atom ID.

In [4]:
for smiles in ['CC', 'CCC', 'CS', 'CSC', 'CF', 'n1ccccc1', '[Br-].Br.I']:
    features = msm.physchem.get_hydrophobic_sites(Chem.MolFromSmiles(smiles))
    print(smiles, features['hydrophobic_atom_indices'])

CC []


CCC [1]


CS []


CSC [1]
CF []


n1ccccc1 [2 3 4]
[Br-].Br.I [1 2]


## Selecting after complete recognition

A selected carbon keeps its original neighborhood for recognition. Removing a nearby nitrogen from the selection does not make that carbon eligible. Chemical matching precedes atom filtering.

In [5]:
pyridine = Chem.MolFromSmiles('n1ccccc1')
for selection in ([1], [2, 3]):
    selected = msm.physchem.get_hydrophobic_sites(pyridine, selection=selection)
    print(selection, selected['hydrophobic_atom_indices'])

[1] []


[2, 3] [2 3]


## Preparing peptide and ligand features

Inspect a peptide with aromatic and aliphatic sidechains and a phenolic ligand before proposing an interface. This exercise concerns declared chemical features, not a binding pose. Backbone heteroatom neighbors exclude some carbon sites; aromaticity alone does not make every ring carbon eligible.

In [6]:
peptide = Chem.MolFromSequence('FLF')
ligand = Chem.MolFromSmiles('CC(C)CCc1ccc(O)cc1')
for label, source in [('peptide', peptide), ('ligand', ligand)]:
    features = msm.physchem.get_hydrophobic_sites(source)
    print(label, features['hydrophobic_atom_indices'])

peptide [ 4  5  6  7  8  9 10 15 16 23 24 25 26 27 28 29]


ligand [ 1  3  4  5  6  7 10 11]


## Keeping hydrogen and state semantics

RDKit interprets indexed and annotated hydrogen counts in the selected state. The tool does not invent hydrogen atoms. Use an explicit stored state or the reference state; structure-assigned states must resolve one state across the requested structures.

In [7]:
implicit = msm.physchem.get_hydrophobic_sites(Chem.MolFromSmiles('CCC.CSC'))
explicit = msm.physchem.get_hydrophobic_sites(Chem.AddHs(Chem.MolFromSmiles('CCC.CSC')))
assert implicit['hydrophobic_atom_indices'].tolist() == explicit['hydrophobic_atom_indices'].tolist()
print(msm.physchem.get_hydrophobic_sites(molsys, chemical_state=0)['chemical_state_index'])

0


## Working with forms and match limits

RDKit, native chemistry, topology and H5MSM supply the same declared-graph contract. Match caps apply before selection and raise instead of silently truncating. An explicit completeness assumption does not repair missing formal charges, bonds or aromatic assignments.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory:
    path = str(Path(directory) / 'chemistry.h5msm')
    msm.convert(molsys, to_form=path)
    from_file = msm.physchem.get_hydrophobic_sites(path, max_matches=100000)
    assert from_file['hydrophobic_atom_indices'].tolist() == sites['hydrophobic_atom_indices'].tolist()
print(sites['evidence'])

{'kind': 'declared_chemical_graph', 'connectivity_completeness': 'complete', 'assume_complete_connectivity': False, 'chemical_state_provenance_index': None}


## Retaining scientific references

The result records the pinned reference implementation and actual MolSysMT/RDKit producers. Optional application-owned Ackredit sessions credit completed recognition, while detached bibliography remains available without Ackredit. This profile is a named interpretation rather than a universal hydrophobicity value.

In [9]:
print(sites['method_reference'])
print(sites['software'])
print([item['id'] for item in sites['attribution']['items']])

{'software': 'ProLIF', 'version': '2.2.2', 'commit': '19f1800218387c49536eb9d3e8cd3044fdb337ee', 'implementation': 'https://github.com/chemosim-lab/ProLIF/blob/19f1800218387c49536eb9d3e8cd3044fdb337ee/prolif/interactions/interactions.py', 'geometry': 'https://github.com/chemosim-lab/ProLIF/blob/19f1800218387c49536eb9d3e8cd3044fdb337ee/prolif/utils.py', 'paper': 'https://doi.org/10.1186/s13321-021-00548-6'}
{'molsysmt': '0.21.0+606.ga03eb4bf6', 'rdkit': '2025.09.5'}
['doi:10.1186/s13321-021-00548-6', 'software:molsysmt:0.21.0+606.ga03eb4bf6', 'software:rdkit:2025.09.5']


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Convert` — conversion with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Get_hydrophobic_interactions` — evaluating typed proximity with {func}`molsysmt.interactions.hydrophobic.get_hydrophobic_interactions`.
- {func}`molsysmt.physchem.get_hydrophobicity` — independent residue scale values.
:::